In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *



In [0]:
try:    initial_load = int(dbutils.widgets.get("initial_load"))
except Exception:
    initial_load = "Dileep"


In [0]:
print(initial_load)

## Data reding from source

In [0]:
df = spark.sql("select * from data_dev.bronze.customers_raw")


removinf duplicates

In [0]:
df=df.dropDuplicates(subset=['c_custkey'])  #This df will be new data frame
df.limit(3).display()

dividing new vs old records

In [0]:
if initial_load == 0:
    df_old=spark.sql("select Dimcustkey,c_custkey, create_date,update_date from data_dev.gold.Dimcustomers")
else:
    df_old=spark.sql("select 0 Dimcustkey,0 c_custkey, 0 create_date, 0 update_date from data_dev.bronze.customers_raw where 1=0")

 

renaming of df_old

In [0]:
df_old=df_old.withColumnRenamed("Dimcustkey","old_Dimcustkey")\
              .withColumnRenamed("c_custkey","old_c_custkey")\
              .withColumnRenamed("create_date","old_create_date")\
              .withColumnRenamed("update_date","old_update_date")
df_old.limit(3).display()


Applying join on old records


In [0]:
df_join = df.join(df_old, df.c_custkey == df_old.old_c_custkey, "left")
df_join.limit(3).display()

seprating old vs new records


In [0]:
df_new = df_join.filter(df_join.old_Dimcustkey.isNull())
df_new.limit(3).display()
df_old = df_join.filter(df_join.old_Dimcustkey.isNotNull())
df_old.limit(3).display()

preparing df_old

In [0]:
#Dropping all the coloumns which are not required
df_old = df_old.drop("old_c_custkey","old_update_date")
#renaming old_Dimcustkey to Dimcustkey
df_old=df_old.withColumnRenamed("old_Dimcustkey","Dimcustkey")
#Renaming old_create_date to create_date
df_old = df_old.withColumnRenamed("old_create_date","create_date")
df_old = df_old.withColumn("create_date", to_timestamp(col("create_date")))


#Recreating update_date with current time stamp
df_old = df_old.withColumn("update_date", current_timestamp())
df_old.limit(3).display()


Preparing df new

In [0]:
#Dropping all the coloumns which are not required
df_new = df_new.drop("old_Dimcustkey","old_c_custkey","old_create_date","old_update_date")


#Recreating update_date and "current date" with current time stamp
df_new = df_new.withColumn("update_date", current_timestamp())
df_new = df_new.withColumn("create_date", current_timestamp())
df_new.limit(3).display()


surrogate key from 1

In [0]:
df_new=df_new.withColumn("Dimcustkey",monotonically_increasing_id()+1)
df_new.limit(3).display()


Adding max surrogate key

In [0]:
if initial_load == 1:
    max_surroagte_key = 0
else:
    df_maxsur = spark.sql("select max(Dimcustkey) as max_surroagte_key from data_dev.gold.Dimcustomers")
    max_surroagte_key = df_maxsur.collect()[0]['max_surroagte_key']



In [0]:
df_new = df_new.withColumn("Dimcustkey",lit(max_surroagte_key)+col("Dimcustkey"))
df_new.limit(3).display()

Applying union on old_df and new_df

In [0]:
df_final=df_new.unionByName(df_old)
df_final.limit(3).display()


## SCD type 1
##### option 1: with the parameter initial_load
##### option 2: with spark catalog if spark.catalog.tableExists(data_dev.gold_Dimcustomers)

In [0]:
from delta.tables import DeltaTable

In [0]:
if initial_load == 0 :
    dlt_obj = DeltaTable.forName(spark, "data_dev.gold.Dimcustomers")
    dlt_obj.alias("trg")\
        .merge(df_final.alias("src"), "trg.Dimcustkey = src.Dimcustkey")\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()
    
else:
    df_final.write.format("delta").mode("overwrite").saveAsTable("data_dev.gold.Dimcustomers")


In [0]:
spark.sql("select * from data_dev.gold.dimcustomers").show()

In [0]:
%sql
select * from data_dev.gold.Dimcustomers